# CR-1a G1 土台の道（60語の頼み方）＋ G1b 費用の見込み（Claude API・CPU）

凍結した CR-1a の修正（`notes/cr1a_amendment.md`）の G1 と G1b です。CR-1 との違いは頼み方（「60語まで」）だけです。

- **G1**：100 のテンプレートすべてを、候補 1 番目から**新しい頼み方で最初から**解かせます。最初の CR-1 の G1 の結果は使いません（保存先も別の `DECISION_POINTS/cr1a/G1`）。
- **G1b**：N ≥ 70 なら、本番の費用を見込み、$90 を超えるならテンプレート番号の大きい方から外します。外したあと N ≥ 70 なら GO。
- 送る前に、頼み方のハッシュが凍結した値と同じかを確かめます。違えば 1 件も送らずに止まります。
- 問題文・応答はどこにも表示しません（数・ID・ハッシュだけ）。

**使い方**
1. ランタイムは **CPU** で大丈夫です。
2. シークレットは `ANTHROPIC_API_KEY` と `GITHUB_TOKEN`。
3. セル1 → セル2 → セル3 の順に実行（セル2 は 10〜20 分くらい）。途中で切れても、もう一度セル1 → セル2 で続きから回ります（同じ実行の記録で、中身の食い違いがないと確かめられたときだけ。結果を保存した候補は呼び直しません。送った直後に切れた 1 件だけは、結果が分からないのでもう一度送り、その数を記録します）。
4. 費用は $0.2〜0.4 の見込み（$3 に近づいたら止まります）。

結果は Drive の `DECISION_POINTS/cr1a/G1/CR1A_G1.json` に入ります。終わったら Claude に教えてね。

- **通知**：スマホの ntfy アプリで `godpork-b06080cfb3cdfa8f` を購読しておくと、始まったとき・終わったとき・エラーで止まったときに届きます。ランタイムごと切れたときは届かないので、そのときは Colab を見てください。
- **終わったら自動でランタイムを切ります**（CU の節約）。Drive への保存を確定してから切るので、結果は残ります。エラーで止まったときは切らずに残します。切りたくないときは、最初に `AUTO_DISCONNECT = False` を実行してください。

In [ ]:
# Cell 1 — Drive, exact head, pinned SDK, key (never printed), notifications, study contract.
from google.colab import drive, userdata
from pathlib import Path
import os, shutil, stat, subprocess, sys, json, time

drive.mount("/content/drive")
STUDY = "CR-1a"
EXPECTED_HEAD = "7d95ee90a7b536fb85326e2a30981b9bc8030421"
BRANCH = "decision-points"
REPO_URL = "https://github.com/chichichi2001/Pork.git"
REPO = Path("/content/Pork")

def get_secret(*names):
    for name in names:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
        if value and str(value).strip():
            return str(value).strip()
    return None

token = get_secret("GITHUB_TOKEN", "GH_TOKEN")
assert token, "Colab Secrets に GITHUB_TOKEN を登録してください。"
askpass = Path("/tmp/pork_git_askpass.sh")
askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) echo "x-access-token" ;;\n  *Password*) echo "$GITHUB_TOKEN" ;;\n  *) echo "" ;;\nesac\n')
askpass.chmod(askpass.stat().st_mode | stat.S_IXUSR)
env = dict(os.environ, GITHUB_TOKEN=token, GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO)], check=True, env=env)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", EXPECTED_HEAD], check=True)
head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert head == EXPECTED_HEAD, (head, EXPECTED_HEAD)
token = None
env.pop("GITHUB_TOKEN", None)
for f in ("notes/cr1_lock_pre_g0.md", "notes/cr1_prompt_record.md", "notes/cr1a_amendment.md", "notes/para1a_amendment.md"):
    assert (REPO / f).exists(), f + " がありません"
sys.path.insert(0, str(REPO))
from pork import cr1_requests as RQ
from pork import cr1_run as R
from pork import cr1_analysis as A
R.check_study(STUDY)                      # prompt hash == the frozen one, or stop before anything is sent
PATHS = R.paths(STUDY)
OUT = Path("/content/drive/MyDrive") / PATHS["dir"]
OUT.mkdir(parents=True, exist_ok=True)
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anthropic==" + RQ.SDK_VERSION], capture_output=True, text=True)
assert r.returncode == 0, r.stderr[-2000:]
import anthropic
assert anthropic.__version__ == RQ.SDK_VERSION, anthropic.__version__
_key = get_secret("ANTHROPIC_API_KEY")
assert _key, "Colab のシークレットに ANTHROPIC_API_KEY を登録してね。"
client = anthropic.Anthropic(api_key=_key, max_retries=0)
del _key
print("study:", STUDY, "| prompt sha256:", RQ.prompt_sha256(STUDY)[:12], "| exact head:", head, "| anthropic", anthropic.__version__)

# 通知（ntfy）：スマホの ntfy アプリでこの名前を購読しておくと、始まったとき・終わったとき・止まったときに届きます。中身は一言だけ（データは送りません）
import urllib.request
NTFY_TOPIC = 'godpork-b06080cfb3cdfa8f'
NB_LABEL = 'CR-1a G1 土台の道＋G1b（60語・API・CPU）'   # どのノートブックからの通知かの名札
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=('[' + NB_LABEL + '] ' + msg).encode('utf-8'),
                                     headers={'Title': 'Pork - CR-1a G1', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        notify('止まりました：エラーが出ています。Colab を見てください', tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

In [ ]:
# Cell 2 — G1 under the study contract: one atomic PRIVATE checkpoint, resumed only if header and content check.
from pork import dp7_matched as d7
from pork import cr1_items as CI
from pork.quality_pork_q2a_gsm8k import parse_gold

rows, DATA_ID = d7.d2.load_rows_from_hub()
chain = CI.chain_and_candidates(rows)
del rows
HDR = R.header(STUDY, head, CI.candidates_sha256(chain), A.MODEL, anthropic.__version__)   # stops if candidates differ from 段2
CKPT = Path("/content/drive/MyDrive") / PATHS["checkpoint"]      # numbers + texts (PRIVATE); the only file read to resume
NUMS = Path("/content/drive/MyDrive") / PATHS["numbers"]         # numbers-only copy, rewritten from CKPT, never read back
assert "/cr1a/" in str(CKPT) and "/cr1/" not in str(CKPT)
R.drive_smoke(CKPT.parent)                 # write -> replace -> reload on this Drive folder, before any request
print("Drive の書き込みの確かめ: OK")
CK = R.load_files(CKPT, NUMS, HDR, chain)    # other run / inconsistent / numbers without checkpoint -> stop, nothing sent
if CK["unknown_outcome"]:
    print("結果が分からないまま切れたリクエスト（もう一度送ります）:", len(CK["unknown_outcome"]))

def save():
    R.save_files(CKPT, NUMS, CK)

RETRY_ON = (anthropic.RateLimitError, anthropic.APIConnectionError, anthropic.InternalServerError, anthropic.APITimeoutError)

print("残りのテンプレート:", sum(str(c["template"]) not in CK["done"] for c in chain), "/", len(chain))
n_calls = R.run_g1(chain, lambda params: client.messages.create(**params), CK, save, parse_gold, retry_on=RETRY_ON)
print("G1 の呼び出しが終わりました。今回の呼び出し:", n_calls, "| 費用（見込み、通算）: $%.3f" % CK["spent_usd"])

In [ ]:
# Cell 3 — G1 gate (N >= 70) and G1b; numbers only.
summary = R.summarize_g1(CK)
summary["dataset"] = DATA_ID
(Path("/content/drive/MyDrive") / PATHS["result"]).write_text(json.dumps(summary, ensure_ascii=False, indent=1))
print(summary["study"], summary["verdict"], "| N（G1 のあと）:", summary["N_after_G1"], "| N（最終）:", summary.get("N_final"))
if "G1b" in summary:
    print("本番の費用の見込み: $%.2f" % summary["G1b"]["estimate"]["usd"])
print("外れた理由の数:", summary["failed_candidates_by_reason"], "| 呼んだ候補:", summary["candidates_called"])
notify('終わりました：CR-1a の G1 と G1b の結果を Drive に書きました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    print('Drive への書き込みを確定してから、ランタイムを切ります（CU の節約。止めたいときは AUTO_DISCONNECT = False）')
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount()
    time.sleep(3)
    _runtime.unassign()